In [3]:
import os
import numpy as np
from PIL import Image

d_original = '/media/fisica/SSD_D0/bandas/banda13/ene_01'
d_reducida = '/media/fisica/SSD_D0/bandas/banda13/ene_01_reducida'

def resize_npy_images(d_original, d_reducida, new_size=(460, 460)):
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            np.save(os.path.join(d_reducida, filename), img_resized)

resize_npy_images(d_original, d_reducida, (460, 460))

In [ ]:
%pip install tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense
import numpy as np
import os

# Directorio de imágenes reducidas
d_reducida = '/media/fisica/SSD_D0/bandas/banda13/ene_01_reducida'

# Cargar imágenes .npy y normalizar según el rango real de radiancia
image_list = []
for filename in os.listdir(d_reducida):
    if filename.lower().endswith('.npy'):
        img = np.load(os.path.join(d_reducida, filename))
        # Normalización robusta: escala entre 0 y 1 usando el rango de radiancia real
        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
        image_list.append(img)

X = np.stack(image_list)
# Añadir canal si es necesario (por ejemplo, imágenes en escala de grises)
if X.ndim == 3:
    X = X[..., np.newaxis]

# Modelo simple con una capa Conv2D
model = Sequential([
    Conv2D(15, (3, 3), activation='relu', padding='same', input_shape=X.shape[1:]), Flatten(),
    Dense(1, activation='sigmoid')  # Cambia según tu tarea
])

model.compile(optimizer='adam', loss='binary_crossentropy')

# Para entrenar necesitas etiquetas (y), aquí solo se muestra cómo preparar X
# model.fit(X, y, epochs=10)

  Using cached tensorflow-2.20.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.32.1-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.75.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.7 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.11.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached ml_dtypes-0.5.3-cp310-cp310-manylinux_2_27_x86_64.manyli

2025-09-16 21:19:00.389943: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-16 21:19:00.444516: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-16 21:19:02.113019: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/fisica/data/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-09-16 21:19:03.639404: E external/l

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 458, 458, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3356224)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │     3,356,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,356,385 (12.80 MB)

 Trainable params: 3,356,385 (12.80 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
from tensorflow.keras.layers import Dense

# Para regresión, cambia la última capa y la función de pérdida
model.pop()  # Elimina la capa Dense actual
model.add(Dense(1, activation='linear'))  # Salida continua

model.compile(optimizer='adam', loss='mse')

# Etiquetas dummy para ejemplo (sustituye por tus valores reales)
y = np.random.rand(X.shape[0], 1)

model.fit(X, y, epochs=10, batch_size=4)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 179ms/step - loss: 41392.5234
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 177ms/step - loss: 13762.5342
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step - loss: 5193.3159
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 179ms/step - loss: 3781.1140
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 179ms/step - loss: 2898.2971
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step - loss: 1199.0806
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step - loss: 447.9142
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 186ms/step - loss: 548.9183
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - loss: 202.7084
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - loss: 108.0544


In [8]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Reshape, Input

# Directorio original y reducido
d_original = '/media/fisica/SSD_D0/bandas/banda13/ene_01'
d_reducida = '/media/fisica/SSD_D0/bandas/banda13/ene_01_reducida'

# Reducción de tamaño a 480x480
def resize_npy_images(d_original, d_reducida, new_size=(480, 480)):
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            np.save(os.path.join(d_reducida, filename), img_resized)

resize_npy_images(d_original, d_reducida, (480, 480))

# Cargar imágenes reducidas y crear pares (img_t, img_t+1)
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
images = [np.load(os.path.join(d_reducida, f)) for f in file_list]
images = [img.astype('float32') / (np.max(img) - np.min(img) + 1e-8) for img in images]

X = np.array(images[:-1])[..., np.newaxis]  # img_t
y = np.array(images[1:])[..., np.newaxis]   # img_t+1

# Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear datasets de TensorFlow
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(4)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(4)

# Modelo simple CNN para predicción de la siguiente imagen

model = Sequential()
model.add(Input(shape=(480, 480, 1)))
model.add(Conv2D(16, (3, 3), activation='relu', padding='same'))
model.add(Flatten())
model.add(Dense(1, activation='linear'))

model.compile(optimizer='adam', loss='mse')
model.fit(train_ds, epochs=10, validation_data=test_ds)

Epoch 1/10


2025-09-16 22:19:09.346220: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [4,480,480,1] vs. [4,1]
	 [[{{function_node __inference_one_step_on_data_2480}}{{node gradient_tape/compile_loss/mse/sub/BroadcastGradientArgs}}]]


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/mse/sub/BroadcastGradientArgs defined at (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/fisica/data/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/fisica/data/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/fisica/data/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/fisica/data/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/usr/lib/python3.10/asyncio/base_events.py", line 1909, in _run_once

  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run

  File "/home/fisica/data/lib/python3.10/site-packages/ipyke

In [ ]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Dropout

# Directorio original y reducido
d_original = '/media/fisica/SSD_D0/bandas/banda13/ene_01'
d_reducida = '/media/fisica/SSD_D0/bandas/banda13/ene_01_reducida'

# Reducción de tamaño a 480x480
def resize_npy_images(d_original, d_reducida, new_size=(480, 480)):
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            np.save(os.path.join(d_reducida, filename), img_resized)

resize_npy_images(d_original, d_reducida, (480, 480))

# Cargar imágenes reducidas y crear pares (img_t, img_t+1)
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
images = [np.load(os.path.join(d_reducida, f)) for f in file_list]

# Normalización global más robusta
all_values = np.concatenate([img.flatten() for img in images])
global_min, global_max = np.percentile(all_values, [1, 99])  # Usar percentiles para robustez
images = [(img - global_min) / (global_max - global_min + 1e-8) for img in images]
images = [np.clip(img, 0, 1) for img in images]  # Asegurar rango [0,1]

X = np.array(images[:-1])[..., np.newaxis]  # img_t
y = np.array(images[1:])[..., np.newaxis]   # img_t+1

print(f"Forma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

# Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear datasets de TensorFlow
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(4).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(4).prefetch(tf.data.AUTOTUNE)

# Modelo Encoder-Decoder para predicción de imágenes
def create_cloud_prediction_model(input_shape=(480, 480, 1)):
    inputs = Input(shape=input_shape)
    
    # Encoder (extrae características espaciales)
    x = Conv2D(32, (7, 7), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(64, (5, 5), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = Dropout(0.3)(x)
    
    # Cuello de botella
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = Dropout(0.3)(x)
    
    # Decoder (reconstruye la imagen predicha)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (5, 5), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(32, (7, 7), activation='relu', padding='same')(x)
    
    # Salida final - misma forma que la entrada
    outputs = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
    
    return Model(inputs, outputs)

# Crear y compilar el modelo
model = create_cloud_prediction_model()

# Optimizador con learning rate adaptativo
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

# Mostrar resumen del modelo
model.summary()

# Callbacks para mejorar el entrenamiento
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

# Entrenar el modelo
history = model.fit(
    train_ds,
    epochs=20,
    validation_data=test_ds,
    callbacks=callbacks,
    verbose=1
)

# Guardar el modelo entrenado
model.save('cloud_prediction_model.h5')

print("Entrenamiento completado!")
print(f"Pérdida final de entrenamiento: {history.history['loss'][-1]:.4f}")
print(f"Pérdida final de validación: {history.history['val_loss'][-1]:.4f}")

Forma de X: (23, 480, 480, 1)
Forma de y: (23, 480, 480, 1)


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 480, 480, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 480, 480, 32)   │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 480, 480, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 480, 480, 64)   │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 480, 480, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 480, 480, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 480, 480, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 480, 480, 256)  │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 480, 480, 256)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 480, 480, 128)  │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 480, 480, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 480, 480, 64)   │       204,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 480, 480, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 480, 480, 32)   │       100,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 480, 480, 1)    │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,023,617 (3.90 MB)

 Trainable params: 1,023,041 (3.90 MB)

 Non-trainable params: 576 (2.25 KB)

Epoch 1/20


2025-09-16 22:25:53.890148: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 235929600 exceeds 10% of free system memory.
2025-09-16 22:25:54.352124: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 235929600 exceeds 10% of free system memory.
2025-09-16 22:25:54.529080: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 235929600 exceeds 10% of free system memory.
2025-09-16 22:25:54.661856: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 471859200 exceeds 10% of free system memory.
2025-09-16 22:25:55.283008: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 471859200 exceeds 10% of free system memory.


5/5 ━━━━━━━━━━━━━━━━━━━━ 325s 62s/step - loss: 0.0640 - mae: 0.1874 - val_loss: 0.0641 - val_mae: 0.1899 - learning_rate: 0.0010
Epoch 2/20
3/5 ━━━━━━━━━━━━━━━━━━━━ 2:16 68s/step - loss: 0.0242 - mae: 0.1158